In [ ]:
# The virtual environment is already created and active
# Install/upgrade pip and requirements
%pip install --upgrade pip
%pip install -r ../requirements.txt

In [ ]:
import torch
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from src.diffusion import add_noise

# Load MNIST image (one sample for demo)
transform = transforms.Compose([transforms.ToTensor()])
mnist = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
x_start, _ = mnist[0]
x_start = x_start.unsqueeze(0)  # Add batch dimension

# Visualize noise progression
T = 10
noisy_imgs = add_noise(x_start, T)

fig, axs = plt.subplots(1, T, figsize=(15, 2))
for i in range(T):
    axs[i].imshow(noisy_imgs[i].squeeze().detach().numpy(), cmap="gray")
    axs[i].axis("off")
    axs[i].set_title(f"t={i}")
plt.tight_layout()
plt.show()

In [ ]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from src.model import UNet
from src.diffusion import get_noise_schedule, q_sample
import os

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([transforms.ToTensor()])
dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Model
model = UNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Diffusion schedule
T = 100
betas = get_noise_schedule(T).to(device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

# Training loop
epochs = 1  # Keep small for demo purposes
model.train()
for epoch in range(epochs):
    for batch in dataloader:
        x, _ = batch
        x = x.to(device)
        t = torch.randint(0, T, (x.size(0),), device=device)
        noise = torch.randn_like(x)
        x_noisy = q_sample(x, t, noise, alphas_cumprod)

        pred_noise = model(x_noisy)
        loss = criterion(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} - Loss: {loss.item():.4f}")

# Save model
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/unet_denoiser.pth")

# Load model (for inference)
model.load_state_dict(torch.load("models/unet_denoiser.pth"))
model.eval()

In [ ]:

import matplotlib.pyplot as plt

def denoise_image(x_start, model, T=100, step=10):
    """
    Adds noise and then attempts to denoise using the trained model.
    Visualizes the original, noisy, and denoised images.
    """
    model.eval()
    device = next(model.parameters()).device
    x_start = x_start.to(device).unsqueeze(0)
    betas = get_noise_schedule(T).to(device)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)

    # Add noise at a fixed step
    t = torch.tensor([step], device=device)
    noise = torch.randn_like(x_start)
    x_noisy = q_sample(x_start, t, noise, alphas_cumprod)

    # Predict noise and denoise
    with torch.no_grad():
        pred_noise = model(x_noisy)
    x_denoised = x_noisy - pred_noise

    # Plot
    imgs = [x_start.squeeze().cpu(), x_noisy.squeeze().cpu(), x_denoised.squeeze().cpu()]
    titles = ["Original", f"Noisy (t={step})", "Denoised"]
    fig, axs = plt.subplots(1, 3, figsize=(9, 3))
    for i in range(3):
        axs[i].imshow(imgs[i], cmap="gray")
        axs[i].set_title(titles[i])
        axs[i].axis("off")
    plt.tight_layout()
    plt.show()

# Example usage
sample_img, _ = dataset[1]
denoise_image(sample_img, model, T=100, step=50)